In [8]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder

ds = load_dataset("ada-datadruids/full_tmdb_movies_dataset")
df = ds["train"].to_pandas()
df

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1142337,741864,Criadores de Melodias - As Lendas do Rock,0.000,0,Released,None,0,0,False,None,...,Criadores de Melodias - As Lendas do Rock,None,0.600,/lZtb2L6E5e0MGBe3iMBl168A8fG.jpg,None,None,None,None,None,None
1142338,741865,Aprenda Trombone Gospel Básico,0.000,0,Released,None,0,0,False,None,...,Aprenda Trombone Gospel Básico,None,0.600,None,None,None,None,None,None,None
1142339,741866,Aprenda Violão Avançado,0.000,0,Released,None,0,0,False,None,...,Aprenda Violão Avançado,None,0.600,/itp1p8Ex8gdSiCcdHfgKcZbhAWw.jpg,None,None,None,None,None,None
1142340,72865,Our Big Sister,0.000,0,Released,1964-04-22,0,0,False,None,...,Our Big Sister,None,0.841,None,None,Drama,None,Hong Kong,Cantonese,None


In [9]:
df['overview'] = df['overview'].fillna('')
df = df.dropna(subset=['title'])

In [10]:
hybrid_embeddings = np.load("model/movie_embeddings_Hybrid_mBERT_XLM.npy")
cross_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
FEATURES = {'genres': 0.15, 'keywords': 0.20, 'production_companies': 0.10}
W_SEMANTIC = 0.55

In [12]:
def to_set(x):
    return set(x) if isinstance(x, (list, set)) else set(str(x).split(','))

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 0.0

In [13]:
def app(title, df, embeddings, top_k_retrieval=500, top_n_final=5):
    match = df[df['title'].str.lower() == title.lower()]
    if match.empty:
        return "Movie not found in the dataset."

    idx = match['vote_count'].idxmax() if len(match) > 1 else match.index[0]

    sims = cosine_similarity(embeddings[idx].reshape(1, -1), embeddings)[0]
    cands = [i for i in sims.argsort()[-top_k_retrieval:][::-1] if i != idx]

    pairs = [[df.loc[idx, 'overview'], df.loc[c, 'overview']] for c in cands]
    cross = cross_model.predict(pairs)
    rng = cross.max() - cross.min()
    cross = (cross - cross.min()) / rng if rng > 0 else cross

    final = W_SEMANTIC * cross
    for col, w in FEATURES.items():
        q = to_set(df.loc[idx, col])
        final += w * np.array([jaccard(q, to_set(df.loc[c, col])) for c in cands])

    top = np.argsort(final)[::-1][:top_n_final]
    result = df.iloc[[cands[i] for i in top]][
        ['title', 'overview', 'genres', 'keywords', 'production_companies', 'release_date']
    ].copy()
    result['final_score'] = final[top]
    return result

In [14]:
app('frozen', df=df, embeddings=hybrid_embeddings)

,title,overview,genres,keywords,production_companies,release_date,final_score
336,Frozen II,"Elsa, Anna, Kristoff and Olaf head far into th...","Family, Animation, Adventure, Comedy, Fantasy","princess, magic, kingdom, winter, queen, castl...","Walt Disney Pictures, Walt Disney Animation St...",2019-11-20,0.606762
2459,Frozen Fever,"On Anna's birthday, Elsa and Kristoff are dete...","Animation, Family, Adventure, Comedy","princess, sibling relationship, birthday, cart...",Walt Disney Animation Studios,2015-03-09,0.559666
31177,Daisy Diamond,A tragic story about Anna who dreams of one th...,Drama,"mother, baby, movie business, film in film, ba...","XX Film, Zentropa Entertainments, Sonet Film, ...",2007-11-23,0.550000
3067,Olaf's Frozen Adventure,Olaf is on a mission to harness the best holid...,"Animation, Family, Adventure, Comedy, Fantasy","holiday, cartoon, snowman, christmas, short fi...",Walt Disney Pictures,2017-10-27,0.525976
57749,King Thrushbeard,A fairy-tale about a beautiful but very haught...,"Fantasy, Family","princess, fairy tale, brothers grimm, beggar",DEFA,1965-12-01,0.513689
